# 05 – Deep Learning (LSTM & Transformer)

Train sequence-to-point LSTM and Transformer encoder models on the credit-spread feature matrix, then compare their performance against the ML baselines.

**Covers:** CreditSpreadDataset, LSTMModel, TransformerModel, training loop with early stopping, RMSE/MAE/directional accuracy comparison.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from config.settings import DEFAULT_START_DATE, DEFAULT_END_DATE, FRED_API_KEY, DATA_DIR, TARGET_HORIZON
from src.data.fetcher import fetch_all_data
from src.features.engineering import build_feature_matrix

df = fetch_all_data(
    start_date=DEFAULT_START_DATE,
    end_date=DEFAULT_END_DATE,
    api_key=FRED_API_KEY,
    cache_dir=DATA_DIR,
)
X, y = build_feature_matrix(df, target_horizon=TARGET_HORIZON)
target_col = [c for c in y.columns if 'return' in c][0]
print(f'Feature matrix: {X.shape}  target: {target_col}')

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X.values)
y_arr = y[target_col].values.astype('float32')
print(f'X_scaled shape: {X_scaled.shape}  |  y shape: {y_arr.shape}')

In [ ]:
from src.models.dl_models import train_dl_model

print('Training LSTM ...')
lstm_result = train_dl_model(
    X_scaled, y_arr,
    model_type='lstm',
    seq_len=20,
    hidden_size=64,
    num_layers=2,
    epochs=50,
    lr=1e-3,
    batch_size=32,
    patience=10,
)
print('LSTM metrics:', lstm_result['metrics'])

In [ ]:
print('Training Transformer ...')
transformer_result = train_dl_model(
    X_scaled, y_arr,
    model_type='transformer',
    seq_len=20,
    hidden_size=64,
    num_layers=2,
    epochs=50,
    lr=1e-3,
    batch_size=32,
    patience=10,
)
print('Transformer metrics:', transformer_result['metrics'])

In [ ]:
# Compare LSTM vs Transformer
comparison = pd.DataFrame({
    'LSTM': lstm_result['metrics'],
    'Transformer': transformer_result['metrics'],
})
print(comparison.round(5))

In [ ]:
# Training loss curves
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(y=lstm_result['train_losses'], mode='lines', name='LSTM Train'))
fig.add_trace(go.Scatter(y=lstm_result['val_losses'], mode='lines', name='LSTM Val', line=dict(dash='dash')))
fig.add_trace(go.Scatter(y=transformer_result['train_losses'], mode='lines', name='Transformer Train'))
fig.add_trace(go.Scatter(y=transformer_result['val_losses'], mode='lines', name='Transformer Val', line=dict(dash='dash')))
fig.update_layout(title='Training Loss Curves', xaxis_title='Epoch', yaxis_title='MSE Loss', template='plotly_white')
fig.show()

In [ ]:
# Forecast vs actual for LSTM
from src.visualization.plots import plot_forecast_vs_actual

n_val = int(len(y_arr) * 0.15)
y_val = y_arr[-n_val:]
preds = lstm_result['predictions']
n_plot = min(len(y_val), len(preds))

fig = plot_forecast_vs_actual(y_val[-n_plot:], preds[-n_plot:], title='LSTM Forecast vs Actual')
fig.show()